# 09 · Evaluate honestly: imbalance, errors, and confidence

**Prerequisites:** train a CNN and distinguish train, validation, and test data.
**Time:** about 35–55 minutes including experiments. Everything uses offline digits.

By the end you will compare macro and aggregate metrics, create an imbalanced
training set without changing evaluation data, and fit temperature scaling on
a separate calibration split. A confidence of 0.9 is a claim about reliability,
not proof that a prediction is correct.

Run each cell in order. Keep the test set closed until the final evaluation.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, classification_report, ConfusionMatrixDisplay)
from course_utils import digit_splits, TinyCNN, evaluate
epochs = 1 if SMOKE_TEST else 8


## 1. Give every split one job

Training updates network weights. Validation selects a model checkpoint and
weighting strategy. Calibration fits one temperature after those decisions.
Test estimates final performance once. We divide the original validation split
into model-validation and calibration subsets with stratification.

This deliberately small calibration set makes uncertainty visible. In a real
project, plan split sizes before collecting results, and split by patient,
subject, product, or recording session when images share a source.


In [ ]:
splits = digit_splits(seed=42)
x_train, y_train = splits['train']
x_hold, y_hold = splits['val']
x_val, x_cal, y_val, y_cal = train_test_split(
    x_hold, y_hold, test_size=0.5, stratify=y_hold, random_state=42)
x_test, y_test = splits['test']
if SMOKE_TEST and len(y_train) > 128:
    x_train, _, y_train, _ = train_test_split(
        x_train, y_train, train_size=128, stratify=y_train, random_state=42)
    x_test, y_test = x_test[:64], y_test[:64]

def make_loader(x, y, shuffle=False):
    ds = TensorDataset(torch.from_numpy(x).unsqueeze(1).float(),
                       torch.from_numpy(y).long())
    return DataLoader(ds, batch_size=64, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(42))

val_loader = make_loader(x_val, y_val)
cal_loader = make_loader(x_cal, y_cal)
test_loader = make_loader(x_test, y_test)
print({'train': len(y_train), 'validation': len(y_val),
       'calibration': len(y_cal), 'test': len(y_test)})


## 2. Create a controlled training imbalance

Keep all examples of digits 0–4 and only one quarter of digits 5–9 in training.
Evaluation retains its natural class distribution. Inverse-frequency weights
are computed **only from retained training labels**. Weighted cross-entropy
increases the contribution of rare-class examples; it does not create new data.


In [ ]:
rng = np.random.default_rng(42)
kept = []
for cls in range(10):
    indices = np.flatnonzero(y_train == cls)
    n = len(indices) if cls < 5 else max(2, len(indices) // 4)
    kept.extend(rng.choice(indices, size=n, replace=False).tolist())
kept = np.array(sorted(kept))
x_imb, y_imb = x_train[kept], y_train[kept]
counts = np.bincount(y_imb, minlength=10)
weights = torch.tensor(len(y_imb) / (10 * counts), dtype=torch.float32)
display(pd.DataFrame({'digit': range(10), 'training_count': counts,
                      'loss_weight': weights.numpy()}))
plt.bar(range(10), counts)
plt.xticks(range(10)); plt.xlabel('Digit'); plt.ylabel('Training images')
plt.show()


In [ ]:
def train_candidate(weighted):
    # Reset initialization and batch order for a controlled comparison.
    seed_everything(42)
    model = TinyCNN().to(device)
    loader = make_loader(x_imb, y_imb, shuffle=True)
    criterion = nn.CrossEntropyLoss(weight=weights.to(device) if weighted else None)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    best_score, best_state, history = -1.0, None, []
    for epoch in range(epochs):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
        result = evaluate(model, val_loader, device=device)
        score = f1_score(result['targets'], result['logits'].argmax(1),
                         labels=list(range(10)), average='macro', zero_division=0)
        history.append({'epoch': epoch + 1, 'val_macro_f1': score})
        if score > best_score:
            best_score, best_state = score, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, history

plain_model, plain_history = train_candidate(False)
weighted_model, weighted_history = train_candidate(True)
print('Training complete; only validation labels selected checkpoints.')


## 3. Read metrics in context

Accuracy gives every example equal weight. Balanced accuracy is mean recall
across classes. Macro F1 computes each class's F1 and averages them equally;
weighted F1 weights class F1 by support. A strong common class can conceal
poor rare-class performance in aggregate scores. F1 and balanced accuracy
answer different questions; neither replaces a per-class review.


In [ ]:
def metric_row(result):
    truth = result['targets'].numpy()
    pred = result['logits'].argmax(1).numpy()
    return {'accuracy': accuracy_score(truth, pred),
            'balanced_accuracy': balanced_accuracy_score(truth, pred),
            'macro_f1': f1_score(truth, pred, labels=list(range(10)),
                                 average='macro', zero_division=0),
            'weighted_f1': f1_score(truth, pred, average='weighted', zero_division=0)}

candidates = {'plain': plain_model, 'weighted': weighted_model}
val_results = {name: evaluate(model, val_loader, device=device)
               for name, model in candidates.items()}
comparison = pd.DataFrame({name: metric_row(result)
                           for name, result in val_results.items()}).T
display(comparison)
chosen_name = comparison['macro_f1'].idxmax()
chosen_model = candidates[chosen_name]
chosen_val = val_results[chosen_name]
print('Predeclared selection metric: validation macro F1. Selected:', chosen_name)


In [ ]:
truth = chosen_val['targets'].numpy()
pred = chosen_val['logits'].argmax(1).numpy()
print(classification_report(truth, pred, labels=list(range(10)), zero_division=0))
ConfusionMatrixDisplay.from_predictions(truth, pred, labels=list(range(10)),
                                        normalize='true', cmap='Blues')
plt.title('Validation: row-normalized confusion matrix')
plt.show()


In [ ]:
wrong = np.flatnonzero(pred != truth)
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax in axes:
    ax.axis('off')
for ax, idx in zip(axes, wrong[:6]):
    ax.imshow(x_val[idx], cmap='gray')
    ax.set_title(f'true {truth[idx]}, predicted {pred[idx]}')
fig.suptitle('Validation errors: investigate patterns before changing a model')
plt.tight_layout(); plt.show()
print('Validation errors:', len(wrong))


## 4. Calibrate the frozen model

Softmax converts logits to probabilities, but training does not guarantee
calibrated confidence. Temperature scaling uses `softmax(logits / T)` with
positive `T`. It changes confidence while preserving the predicted class.

Freeze the network. Fit `T` by minimizing cross-entropy on calibration data
only. We constrain `T` to [0.05, 20] for numerical stability in this tiny
experiment. A boundary optimum is a reason to investigate sample size and
model quality, not to claim successful calibration.


In [ ]:
import torch.nn.functional as F
cal_result = evaluate(chosen_model, cal_loader, device=device)
# evaluate uses inference_mode; clone outside it before autograd fits T.
cal_logits = cal_result['logits'].detach().clone().float()
cal_targets = cal_result['targets'].clone()
log_temperature = nn.Parameter(torch.zeros(()))
temperature_optimizer = torch.optim.LBFGS([log_temperature], lr=0.2, max_iter=40)

def temperature_value():
    return log_temperature.exp().clamp(0.05, 20.0)

def closure():
    temperature_optimizer.zero_grad()
    loss = F.cross_entropy(cal_logits / temperature_value(), cal_targets)
    loss.backward()
    return loss

temperature_optimizer.step(closure)
temperature = temperature_value().detach().item()
print(f'Temperature: {temperature:.3f}')
print('Calibration NLL before:', F.cross_entropy(cal_logits, cal_targets).item())
print('Calibration NLL after: ', F.cross_entropy(cal_logits / temperature, cal_targets).item())


## 5. Understand a reliability diagram

We group examples by their **top predicted probability**. For each bin, compare
mean confidence with the fraction correctly classified. This is top-label
calibration, not a complete classwise calibration test.

Expected calibration error (ECE) is a bin-count-weighted average absolute gap.
It depends on bin boundaries, sample size, and the confidence distribution.
A tiny ECE can coexist with a weak classifier. Always inspect bin counts and
negative log-likelihood (NLL); calibration cannot repair incorrect class rankings.


In [ ]:
def reliability(logits, labels, bins=8):
    probabilities = logits.softmax(1)
    confidence, prediction = probabilities.max(1)
    correct = prediction.eq(labels).float()
    bin_id = torch.clamp((confidence * bins).long(), max=bins - 1)
    rows = []
    for b in range(bins):
        mask = bin_id == b
        count = int(mask.sum())
        rows.append({'bin': b, 'count': count,
                     'confidence': confidence[mask].mean().item() if count else np.nan,
                     'accuracy': correct[mask].mean().item() if count else np.nan})
    table = pd.DataFrame(rows)
    ece = ((table['count'] / len(labels)) *
           (table['confidence'] - table['accuracy']).abs()).sum()
    return table, float(ece)

# Calibration-set graphs are diagnostics, not an unbiased final score.
before, ece_before = reliability(cal_logits, cal_targets)
after, ece_after = reliability(cal_logits / temperature, cal_targets)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for table, label in [(before, 'Original'), (after, 'Temperature-scaled')]:
    used = table['count'] > 0
    axes[0].plot(table.loc[used, 'confidence'], table.loc[used, 'accuracy'], 'o-', label=label)
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set(xlabel='Mean confidence', ylabel='Observed accuracy', title='Calibration split')
axes[0].legend()
axes[1].bar(after['bin'], after['count'])
axes[1].set(xlabel='Confidence bin', ylabel='Images', title='Scaled confidence support')
plt.tight_layout(); plt.show()
print({'calibration_ece_before': ece_before, 'calibration_ece_after': ece_after})


## 6. Open the test set once

The model, checkpoint, weighting choice, temperature method, and bins are now
fixed. One forward pass provides logits for both original and scaled
probabilities. Report both; do not choose the method after seeing test results.
Calibration may help NLL while ECE gets worse, especially with few examples.


In [ ]:
test_result = evaluate(chosen_model, test_loader, device=device)
test_logits, test_targets = test_result['logits'], test_result['targets']
rows = []
for name, scale in [('original', 1.0), ('temperature_scaled', temperature)]:
    logits = test_logits / scale
    _, ece = reliability(logits, test_targets)
    rows.append({'method': name, **metric_row({'logits': logits, 'targets': test_targets}),
                 'nll': F.cross_entropy(logits, test_targets).item(), 'ece_8_bins': ece})
final_report = pd.DataFrame(rows)
display(final_report)
assert torch.equal(test_logits.argmax(1), (test_logits / temperature).argmax(1))
out_dir = ARTIFACTS / '09_evaluation'; out_dir.mkdir(parents=True, exist_ok=True)
final_report.to_csv(out_dir / 'test_metrics.csv', index=False)
print('Saved:', out_dir / 'test_metrics.csv')


## Practice 1 · Why can accuracy be misleading?

Change the number of majority-class examples below. Predict accuracy and
balanced accuracy before running. This uses synthetic labels, so it leaves
your test protocol intact.


In [ ]:
majority_count = 90  # Try 50 and 990.
synthetic_truth = np.array([0] * majority_count + [1] * 10)
always_zero = np.zeros_like(synthetic_truth)
print('Accuracy:', accuracy_score(synthetic_truth, always_zero))
print('Balanced accuracy:', balanced_accuracy_score(synthetic_truth, always_zero))


<details><summary>Solution idea</summary>
Accuracy is majority_count / (majority_count + 10), while balanced accuracy
stays at 0.5 because class 0 recall is 1 and class 1 recall is 0.
</details>

## Practice 2 · Change ECE bins

Recompute ECE using calibration logits with 3, 5, 10, and 20 bins. Keep the
temperature fixed. Explain why choosing whichever bin count looks best would
misrepresent calibration quality.


In [ ]:
bin_experiment = []
for bins in [3, 5, 10, 20]:
    table, score = reliability(cal_logits / temperature, cal_targets, bins=bins)
    bin_experiment.append({'bins': bins, 'ece': score,
                           'nonempty_bins': int((table['count'] > 0).sum())})
display(pd.DataFrame(bin_experiment))


<details><summary>Solution idea</summary>
Bin boundaries regroup examples, altering both gaps and estimation noise.
Report a predefined scheme with support counts and NLL, and use enough data.
</details>

## Practice 3 · Explore abstention on validation data

A classifier can abstain when confidence is low. Change `threshold` and
measure coverage (fraction answered) and accuracy among answers. Use
validation to design a rule; evaluating a new final rule needs a fresh
held-out test because this notebook has already revealed its test results.


In [ ]:
threshold = 0.6  # Try 0.3 and 0.8.
confidence, prediction = (chosen_val['logits'] / temperature).softmax(1).max(1)
accepted = confidence >= threshold
selective_accuracy = (prediction[accepted] == chosen_val['targets'][accepted]).float().mean().item() if accepted.any() else float('nan')
print({'threshold': threshold, 'coverage': accepted.float().mean().item(),
       'accuracy_when_accepted': selective_accuracy, 'accepted_images': int(accepted.sum())})


<details><summary>Solution idea</summary>
A higher threshold often reduces coverage; accuracy need not improve
monotonically on a small sample. No accepted examples means undefined
selective accuracy, not perfect accuracy. Inspect classwise coverage too.
</details>

**Completion check:** explain the four split roles, read a confusion-matrix
row, compute inverse-frequency weights, and explain why temperature preserves
argmax. Weighted training and calibration are useful tools, not guaranteed wins.

**Primary sources:** [scikit-learn classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics),
[probability calibration](https://scikit-learn.org/stable/modules/calibration.html),
[Guo et al., temperature scaling](https://proceedings.mlr.press/v70/guo17a.html).

**Next:** [10 · Experiments and checkpoints](10_experiments_checkpoints.ipynb).
